# Model Validation with FLORES-200 Dataset

This notebook validates the trained NLLB-200 models with LoRA adapters using the FLORES-200 benchmark dataset.

## Validation Setup
- **Base Model:** NLLB-200-distilled-600M
- **Fine-tuning Method:** LoRA adapters with BF16 mixed precision
- **Benchmark Dataset:** FLORES-200 devtest split
- **Metrics:** BLEU score (sacrebleu)

## Models to Validate
1. **Baseline Models:** Direct fine-tuning (e.g., en→tl)
2. **Experimental Stage 1:** Similar language transfer (e.g., war→tl)
3. **Experimental Stage 2:** Sequential fine-tuning (war→tl THEN en→tl)

## Expected Results
Compare baseline vs experimental Stage 2 to measure the impact of similarity transfer.

In [1]:
# ============================================================================
# CRITICAL FIX FOR WINDOWS UTF-8 ENCODING ISSUE
# This MUST be the first code cell and kernel MUST be restarted after adding this
# ============================================================================

import sys
import os

# Patch built-in open() to default to UTF-8
import builtins
_original_open = builtins.open

def utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, 
              newline=None, closefd=True, opener=None):
    if encoding is None and 'b' not in str(mode):
        encoding = 'utf-8'
    return _original_open(file, mode, buffering, encoding, errors, newline, closefd, opener)

builtins.open = utf8_open

# Set environment variables (helps for subprocesses)
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

print("✓ UTF-8 encoding patch applied")
print(f"  System: {sys.platform}")

# NOW import other libraries
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from datasets import load_dataset
import evaluate
import pandas as pd
from tqdm import tqdm
import json
from pathlib import Path

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

✓ UTF-8 encoding patch applied
  System: win32


W1119 03:30:55.586000 14396 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



Using device: cuda
PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3050
BF16 supported: True


## Configuration

Set up paths, language codes, and models to evaluate.

In [ ]:
# Base model and directories
BASE_MODEL = "facebook/nllb-200-distilled-600M"
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# NLLB-200 language codes (matching training notebook)
NLLB_LANG_CODES = {
    "en": "eng_Latn",
    "tl": "tgl_Latn",  # Tagalog
    "war": "war_Latn",  # Waray
    "ceb": "ceb_Latn",  # Cebuano
}

# Mapping from FLORES dataset keys to NLLB codes
FLORES_TO_NLLB = {
    "eng_Latn": "eng_Latn",
    "tgl_Latn": "tgl_Latn",
    "ceb_Latn": "ceb_Latn",
    "war_Latn": "war_Latn",
}

# Define models to evaluate - matching training notebook structure
MODELS_TO_EVALUATE = {
    #"tagalog": {
    #    "baseline": MODELS_DIR / "tagalog_baseline_nllb_lora_bf16" / "final_model",
    #    "experimental_stage1": MODELS_DIR / "tagalog_experimental_stage1_nllb_lora_bf16" / "final_model",
    #    "experimental_stage2": MODELS_DIR / "tagalog_experimental_stage2_nllb_lora_bf16" / "final_model",
    #    "src_lang": "eng_Latn",
    #    "tgt_lang": "tgl_Latn",
    #    "flores_src": "eng_Latn",
    #    "flores_tgt": "tgl_Latn",
    #},
    "waray": {
        "baseline": MODELS_DIR / "waray_baseline_nllb_lora_bf16" / "final_model",
        "experimental_stage1": MODELS_DIR / "waray_experimental_stage1_nllb_lora_bf16" / "final_model",
        "experimental_stage2": MODELS_DIR / "waray_experimental_stage2_nllb_lora_bf16" / "final_model",
        "src_lang": "eng_Latn",
        "tgt_lang": "war_Latn",
        "flores_src": "eng_Latn",
        "flores_tgt": "war_Latn",
    }
}

print("Configuration:")
print(f"  Base Model: {BASE_MODEL}")
print(f"  Models Directory: {MODELS_DIR}")
print(f"  Results Directory: {RESULTS_DIR}")
print(f"\nTarget Languages:")
for lang, config in MODELS_TO_EVALUATE.items():
    print(f"  - {lang.capitalize()}: {config['src_lang']} → {config['tgt_lang']}")
    print(f"    Baseline: {config['baseline'].exists()}")
    print(f"    Stage 1:  {config['experimental_stage1'].exists()}")
    print(f"    Stage 2:  {config['experimental_stage2'].exists()}")

Configuration:
  Base Model: facebook/nllb-200-distilled-600M
  Models Directory: ..\models
  Results Directory: ..\results

Target Languages:
  - Tagalog: eng_Latn → tgl_Latn
    Baseline: True
    Stage 1:  True
    Stage 2:  True
  - Waray: eng_Latn → war_Latn
    Baseline: False
    Stage 1:  False
    Stage 2:  False


## Load FLORES-200 Dataset

Load the FLORES-200 benchmark dataset for evaluation.

In [3]:
# Load FLORES-200 dataset
# Note: FLORES-200 loads each language separately, not as "all"
print("Loading FLORES-200 dataset...")

# We'll load languages on-demand during extraction
# First, verify the dataset is accessible

# Test load one language to verify access
test_dataset = load_dataset("facebook/flores", "eng_Latn", split="devtest")
print(f"✓ FLORES-200 dataset accessible!")
print(f"  Dataset: facebook/flores")
print(f"  Split: devtest")
print(f"  Number of examples per language: {len(test_dataset)}")
print(f"  Each language will be loaded separately during extraction")


Loading FLORES-200 dataset...
✓ FLORES-200 dataset accessible!
  Dataset: facebook/flores
  Split: devtest
  Number of examples per language: 1012
  Each language will be loaded separately during extraction
✓ FLORES-200 dataset accessible!
  Dataset: facebook/flores
  Split: devtest
  Number of examples per language: 1012
  Each language will be loaded separately during extraction


## Extract Test Data

Extract source and reference sentences for each language pair.

In [4]:
# Extract test data by loading each language separately
# FLORES-200 structure: load_dataset("facebook/flores", "lang_code", split="devtest")
# Each example has: {'id': int, 'URL': str, 'domain': str, 'topic': str, 'has_image': str, 'sentence': str}

print("\nExtracting test data from FLORES-200...")
test_data = {}

for target_lang, config in MODELS_TO_EVALUATE.items():
    src_lang = config['flores_src']
    tgt_lang = config['flores_tgt']
    
    try:
        print(f"\n{target_lang.capitalize()}: {src_lang} → {tgt_lang}")
        
        # Load source language dataset
        print(f"  Loading {src_lang}...")
        src_dataset = load_dataset("facebook/flores", src_lang, split="devtest")
        src_sentences = [example['sentence'] for example in src_dataset]
        
        # Load target language dataset
        print(f"  Loading {tgt_lang}...")
        tgt_dataset = load_dataset("facebook/flores", tgt_lang, split="devtest")
        tgt_sentences = [example['sentence'] for example in tgt_dataset]
        
        # Verify same number of sentences
        if len(src_sentences) != len(tgt_sentences):
            print(f"  ⚠️  Warning: Mismatched lengths ({len(src_sentences)} vs {len(tgt_sentences)})")
        
        # Store data
        pair_key = f"{src_lang}-{tgt_lang}"
        test_data[pair_key] = {
            "source": src_sentences,
            "reference": tgt_sentences
        }
        print(f"  ✓ Extracted {len(src_sentences)} sentence pairs")
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        print(f"     Skipping {target_lang}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"✓ Test data extraction complete!")
print(f"  Available language pairs: {list(test_data.keys())}")
print(f"  Total pairs extracted: {len(test_data)}")
print("="*80)


Extracting test data from FLORES-200...

Tagalog: eng_Latn → tgl_Latn
  Loading eng_Latn...
  Loading tgl_Latn...
  Loading tgl_Latn...
  ✓ Extracted 1012 sentence pairs

Waray: eng_Latn → war_Latn
  Loading eng_Latn...
  ✓ Extracted 1012 sentence pairs

Waray: eng_Latn → war_Latn
  Loading eng_Latn...
  Loading war_Latn...
  Loading war_Latn...
  ✓ Extracted 1012 sentence pairs

✓ Test data extraction complete!
  Available language pairs: ['eng_Latn-tgl_Latn', 'eng_Latn-war_Latn']
  Total pairs extracted: 2
  ✓ Extracted 1012 sentence pairs

✓ Test data extraction complete!
  Available language pairs: ['eng_Latn-tgl_Latn', 'eng_Latn-war_Latn']
  Total pairs extracted: 2


## Load Evaluation Metric

Load the BLEU metric for evaluation (matching training notebook).

In [5]:
# Load BLEU metric (sacrebleu)
bleu_metric = evaluate.load("sacrebleu")
print("✓ BLEU metric (sacrebleu) loaded successfully")

✓ BLEU metric (sacrebleu) loaded successfully


## Translation Function

Function to translate sentences using NLLB-200 with LoRA adapters (matching training notebook).

In [6]:
def translate_batch(model, tokenizer, sentences, src_lang, tgt_lang, batch_size=8, max_length=128):
    """Translate a batch of sentences using NLLB-200 with LoRA.
    
    Args:
        model: NLLB model with LoRA adapters
        tokenizer: NLLB tokenizer
        sentences: List of source sentences
        src_lang: Source language code (e.g., 'eng_Latn')
        tgt_lang: Target language code (e.g., 'tgl_Latn')
        batch_size: Batch size for translation
        max_length: Maximum sequence length
    
    Returns:
        List of translated sentences
    """
    translations = []
    
    # Set source language for tokenizer
    tokenizer.src_lang = src_lang
    
    # Process in batches
    for i in tqdm(range(0, len(sentences), batch_size), desc="Translating"):
        batch = sentences[i:i+batch_size]
        
        # Tokenize source sentences
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate translations
        # Force target language as BOS token
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=max_length,
                num_beams=5,
                early_stopping=True
            )
        
        # Decode translations
        batch_translations = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )
        translations.extend(batch_translations)
    
    return translations

print("✓ Translation function defined")

✓ Translation function defined


## Evaluation Function

Function to evaluate a model and compute BLEU scores (matching training notebook).

In [7]:
def evaluate_model(model_name, model_path, test_data, src_lang, tgt_lang):
    """Evaluate a model on FLORES-200 test data.
    
    Args:
        model_name: Name of the model (for reporting)
        model_path: Path to model with LoRA adapters
        test_data: Dictionary with 'source' and 'reference' lists
        src_lang: Source language code
        tgt_lang: Target language code
    
    Returns:
        Dictionary with evaluation results
    """
    print("\n" + "="*80)
    print(f"Evaluating: {model_name}")
    print(f"Model path: {model_path}")
    print(f"Language pair: {src_lang} → {tgt_lang}")
    print("="*80 + "\n")
    
    # Check if model exists
    if not model_path.exists():
        print(f"❌ Model not found: {model_path}")
        return None
    
    # Load tokenizer and model
    try:
        print("1. Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        
        print("2. Loading base model...")
        base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
        
        print("3. Loading LoRA adapters...")
        model = PeftModel.from_pretrained(base_model, str(model_path))
        
        print("4. Moving to device...")
        model.to(device)
        model.eval()
        
        print("✓ Model loaded successfully\n")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # Get test sentences
    source_sentences = test_data["source"]
    reference_sentences = test_data["reference"]
    
    print(f"5. Translating {len(source_sentences)} sentences...")
    translations = translate_batch(
        model, tokenizer, source_sentences, src_lang, tgt_lang
    )
    
    # Calculate BLEU score
    print("\n6. Computing BLEU score...")
    # sacrebleu expects references as list of lists
    references = [[ref] for ref in reference_sentences]
    
    bleu_results = bleu_metric.compute(
        predictions=translations,
        references=references
    )
    
    # Clean up memory
    del model
    del base_model
    torch.cuda.empty_cache()
    
    # Prepare results
    results = {
        "model_name": model_name,
        "model_path": str(model_path),
        "src_lang": src_lang,
        "tgt_lang": tgt_lang,
        "bleu_score": bleu_results["score"],
        "bleu_precisions": bleu_results["precisions"],
        "brevity_penalty": bleu_results["bp"],
        "length_ratio": bleu_results["sys_len"] / bleu_results["ref_len"],
        "translation_length": bleu_results["sys_len"],
        "reference_length": bleu_results["ref_len"],
        "num_sentences": len(translations),
        "sample_translations": [
            {
                "source": source_sentences[i],
                "reference": reference_sentences[i],
                "translation": translations[i]
            } for i in range(min(5, len(translations)))
        ]
    }
    
    print(f"\n✓ Evaluation complete!")
    print(f"  BLEU Score: {bleu_results['score']:.2f}")
    print(f"  Precisions: {[f'{p:.2f}' for p in bleu_results['precisions']]}")
    print(f"  Brevity Penalty: {bleu_results['bp']:.4f}")
    
    return results

print("✓ Evaluation function defined")

✓ Evaluation function defined


## Run Evaluation

Evaluate all trained models on FLORES-200 test set.

In [8]:
# Run evaluation for all models
all_results = {}

print("\n" + "#"*80)
print("# FLORES-200 EVALUATION")
print("#"*80)

for target_lang, config in MODELS_TO_EVALUATE.items():
    print(f"\n{'='*80}")
    print(f"Target Language: {target_lang.upper()}")
    print(f"Language Pair: {config['src_lang']} → {config['tgt_lang']}")
    print("="*80)
    
    # Get test data for this language pair
    pair_key = f"{config['flores_src']}-{config['flores_tgt']}"
    if pair_key not in test_data:
        print(f"❌ No test data available for {pair_key}")
        continue
    
    lang_test_data = test_data[pair_key]
    lang_results = {}
    
    # Evaluate baseline model
    if config['baseline'].exists():
        result = evaluate_model(
            model_name=f"{target_lang.capitalize()} - Baseline",
            model_path=config['baseline'],
            test_data=lang_test_data,
            src_lang=config['src_lang'],
            tgt_lang=config['tgt_lang']
        )
        if result:
            lang_results['baseline'] = result
    else:
        print(f"\n⚠️  Baseline model not found: {config['baseline']}")
    
    # Evaluate experimental stage 1 model
    if config['experimental_stage1'].exists():
        result = evaluate_model(
            model_name=f"{target_lang.capitalize()} - Experimental Stage 1",
            model_path=config['experimental_stage1'],
            test_data=lang_test_data,
            src_lang=config['src_lang'],
            tgt_lang=config['tgt_lang']
        )
        if result:
            lang_results['experimental_stage1'] = result
    else:
        print(f"\n⚠️  Experimental Stage 1 model not found: {config['experimental_stage1']}")
    
    # Evaluate experimental stage 2 model
    if config['experimental_stage2'].exists():
        result = evaluate_model(
            model_name=f"{target_lang.capitalize()} - Experimental Stage 2",
            model_path=config['experimental_stage2'],
            test_data=lang_test_data,
            src_lang=config['src_lang'],
            tgt_lang=config['tgt_lang']
        )
        if result:
            lang_results['experimental_stage2'] = result
    else:
        print(f"\n⚠️  Experimental Stage 2 model not found: {config['experimental_stage2']}")
    
    # Store results for this language
    if lang_results:
        all_results[target_lang] = lang_results

print("\n" + "#"*80)
print(f"# EVALUATION COMPLETE - {len(all_results)} language(s) evaluated")
print("#"*80)


################################################################################
# FLORES-200 EVALUATION
################################################################################

Target Language: TAGALOG
Language Pair: eng_Latn → tgl_Latn

Evaluating: Tagalog - Baseline
Model path: ..\models\tagalog_baseline_nllb_lora_bf16\final_model
Language pair: eng_Latn → tgl_Latn

1. Loading tokenizer...
2. Loading base model...
3. Loading LoRA adapters...
4. Moving to device...
✓ Model loaded successfully

5. Translating 1012 sentences...


Translating: 100%|██████████| 127/127 [06:21<00:00,  3.00s/it]



6. Computing BLEU score...

✓ Evaluation complete!
  BLEU Score: 31.29
  Precisions: ['65.95', '38.81', '24.96', '16.45']
  Brevity Penalty: 0.9772

Evaluating: Tagalog - Experimental Stage 1
Model path: ..\models\tagalog_experimental_stage1_nllb_lora_bf16\final_model
Language pair: eng_Latn → tgl_Latn

1. Loading tokenizer...
2. Loading base model...
3. Loading LoRA adapters...
4. Moving to device...
✓ Model loaded successfully

5. Translating 1012 sentences...


Translating: 100%|██████████| 127/127 [06:14<00:00,  2.95s/it]



6. Computing BLEU score...

✓ Evaluation complete!
  BLEU Score: 30.69
  Precisions: ['66.06', '38.72', '24.75', '16.16']
  Brevity Penalty: 0.9649

Evaluating: Tagalog - Experimental Stage 2
Model path: ..\models\tagalog_experimental_stage2_nllb_lora_bf16\final_model
Language pair: eng_Latn → tgl_Latn

1. Loading tokenizer...
2. Loading base model...
3. Loading LoRA adapters...
4. Moving to device...
✓ Model loaded successfully

5. Translating 1012 sentences...


Translating: 100%|██████████| 127/127 [06:10<00:00,  2.92s/it]


6. Computing BLEU score...

✓ Evaluation complete!
  BLEU Score: 31.46
  Precisions: ['65.65', '38.58', '24.84', '16.40']
  Brevity Penalty: 0.9871

Target Language: WARAY
Language Pair: eng_Latn → war_Latn

⚠️  Baseline model not found: ..\models\waray_baseline_nllb_lora_bf16\final_model

⚠️  Experimental Stage 1 model not found: ..\models\waray_experimental_stage1_nllb_lora_bf16\final_model

⚠️  Experimental Stage 2 model not found: ..\models\waray_experimental_stage2_nllb_lora_bf16\final_model

################################################################################
# EVALUATION COMPLETE - 1 language(s) evaluated
################################################################################


## Evaluate Base Model (No Fine-tuning)

Evaluate the pretrained NLLB-200 model without any fine-tuning as a baseline comparison.

In [9]:
def evaluate_base_model(model_name, test_data, src_lang, tgt_lang):
    """Evaluate the pretrained base NLLB-200 model (no fine-tuning).
    
    Args:
        model_name: Name of the model (for reporting)
        test_data: Dictionary with 'source' and 'reference' lists
        src_lang: Source language code
        tgt_lang: Target language code
    
    Returns:
        Dictionary with evaluation results
    """
    print("\n" + "="*80)
    print(f"Evaluating: {model_name}")
    print(f"Model: {BASE_MODEL} (Pretrained - No Fine-tuning)")
    print(f"Language pair: {src_lang} → {tgt_lang}")
    print("="*80 + "\n")
    
    # Load tokenizer and model
    try:
        print("1. Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        
        print("2. Loading base model...")
        model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
        
        print("3. Moving to device...")
        model.to(device)
        model.eval()
        
        print("✓ Model loaded successfully\n")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # Get test sentences
    source_sentences = test_data["source"]
    reference_sentences = test_data["reference"]
    
    print(f"4. Translating {len(source_sentences)} sentences...")
    translations = translate_batch(
        model, tokenizer, source_sentences, src_lang, tgt_lang
    )
    
    # Calculate BLEU score
    print("\n5. Computing BLEU score...")
    # sacrebleu expects references as list of lists
    references = [[ref] for ref in reference_sentences]
    
    bleu_results = bleu_metric.compute(
        predictions=translations,
        references=references
    )
    
    # Clean up memory
    del model
    torch.cuda.empty_cache()
    
    # Prepare results
    results = {
        "model_name": model_name,
        "model_path": "pretrained (no fine-tuning)",
        "src_lang": src_lang,
        "tgt_lang": tgt_lang,
        "bleu_score": bleu_results["score"],
        "bleu_precisions": bleu_results["precisions"],
        "brevity_penalty": bleu_results["bp"],
        "length_ratio": bleu_results["sys_len"] / bleu_results["ref_len"],
        "translation_length": bleu_results["sys_len"],
        "reference_length": bleu_results["ref_len"],
        "num_sentences": len(translations),
        "sample_translations": [
            {
                "source": source_sentences[i],
                "reference": reference_sentences[i],
                "translation": translations[i]
            } for i in range(min(5, len(translations)))
        ]
    }
    
    print(f"\n✓ Evaluation complete!")
    print(f"  BLEU Score: {bleu_results['score']:.2f}")
    print(f"  Precisions: {[f'{p:.2f}' for p in bleu_results['precisions']]}")
    print(f"  Brevity Penalty: {bleu_results['bp']:.4f}")
    
    return results


# Evaluate base model for each language pair
print("\n" + "#"*80)
print("# BASE MODEL EVALUATION (NO FINE-TUNING)")
print("#"*80)

base_model_results = {}

for target_lang, config in MODELS_TO_EVALUATE.items():
    print(f"\n{'='*80}")
    print(f"Target Language: {target_lang.upper()}")
    print(f"Language Pair: {config['src_lang']} → {config['tgt_lang']}")
    print("="*80)
    
    # Get test data for this language pair
    pair_key = f"{config['flores_src']}-{config['flores_tgt']}"
    if pair_key not in test_data:
        print(f"❌ No test data available for {pair_key}")
        continue
    
    lang_test_data = test_data[pair_key]
    
    # Evaluate base model
    result = evaluate_base_model(
        model_name=f"{target_lang.capitalize()} - Base Model (Pretrained)",
        test_data=lang_test_data,
        src_lang=config['src_lang'],
        tgt_lang=config['tgt_lang']
    )
    
    if result:
        base_model_results[target_lang] = result

print("\n" + "#"*80)
print(f"# BASE MODEL EVALUATION COMPLETE - {len(base_model_results)} language(s) evaluated")
print("#"*80)

# Display base model results
print("\n" + "="*80)
print("BASE MODEL RESULTS (Pretrained NLLB-200, No Fine-tuning)")
print("="*80)
for target_lang, result in base_model_results.items():
    print(f"\n{target_lang.upper()}:")
    print(f"  BLEU Score: {result['bleu_score']:.2f}")
    print(f"  Language Pair: {result['src_lang']} → {result['tgt_lang']}")
print("="*80)


################################################################################
# BASE MODEL EVALUATION (NO FINE-TUNING)
################################################################################

Target Language: TAGALOG
Language Pair: eng_Latn → tgl_Latn

Evaluating: Tagalog - Base Model (Pretrained)
Model: facebook/nllb-200-distilled-600M (Pretrained - No Fine-tuning)
Language pair: eng_Latn → tgl_Latn

1. Loading tokenizer...
2. Loading base model...
3. Moving to device...
✓ Model loaded successfully

4. Translating 1012 sentences...


Translating: 100%|██████████| 127/127 [05:57<00:00,  2.82s/it]



5. Computing BLEU score...

✓ Evaluation complete!
  BLEU Score: 30.74
  Precisions: ['66.25', '38.88', '24.90', '16.34']
  Brevity Penalty: 0.9607

Target Language: WARAY
Language Pair: eng_Latn → war_Latn

Evaluating: Waray - Base Model (Pretrained)
Model: facebook/nllb-200-distilled-600M (Pretrained - No Fine-tuning)
Language pair: eng_Latn → war_Latn

1. Loading tokenizer...
2. Loading base model...
3. Moving to device...
✓ Model loaded successfully

4. Translating 1012 sentences...


Translating:  61%|██████▏   | 78/127 [04:06<02:34,  3.15s/it]


KeyboardInterrupt: 

## Display Results

Show evaluation results in a summary table.

In [14]:
# Prepare results for display
results_list = []

# Add base model results first
for target_lang, result in base_model_results.items():
    results_list.append({
        "Target Language": target_lang.capitalize(),
        "Model": "Base Model (Pretrained)",
        "BLEU Score": f"{result['bleu_score']:.2f}",
        "BLEU-1": f"{result['bleu_precisions'][0]:.2f}",
        "BLEU-2": f"{result['bleu_precisions'][1]:.2f}",
        "BLEU-3": f"{result['bleu_precisions'][2]:.2f}",
        "BLEU-4": f"{result['bleu_precisions'][3]:.2f}",
        "BP": f"{result['brevity_penalty']:.4f}",
    })

# Add fine-tuned model results
for target_lang, lang_results in all_results.items():
    for stage, result in lang_results.items():
        results_list.append({
            "Target Language": target_lang.capitalize(),
            "Model": stage.replace('_', ' ').title(),
            "BLEU Score": f"{result['bleu_score']:.2f}",
            "BLEU-1": f"{result['bleu_precisions'][0]:.2f}",
            "BLEU-2": f"{result['bleu_precisions'][1]:.2f}",
            "BLEU-3": f"{result['bleu_precisions'][2]:.2f}",
            "BLEU-4": f"{result['bleu_precisions'][3]:.2f}",
            "BP": f"{result['brevity_penalty']:.4f}",
        })

# Create DataFrame
results_df = pd.DataFrame(results_list)

# Display table
print("\n" + "="*100)
print("FLORES-200 EVALUATION RESULTS (NLLB-200 + LoRA + BF16)")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

# Calculate improvements
print("\n" + "="*100)
print("IMPROVEMENT ANALYSIS")
print("="*100)

for target_lang, lang_results in all_results.items():
    print(f"\n{target_lang.upper()}:")
    
    # Compare with base model
    if target_lang in base_model_results:
        base_model_bleu = base_model_results[target_lang]['bleu_score']
        print(f"  Base Model (Pretrained):   {base_model_bleu:.2f} BLEU")
    
    # Show all fine-tuned models
    if 'baseline' in lang_results:
        baseline_bleu = lang_results['baseline']['bleu_score']
        print(f"  Baseline:                  {baseline_bleu:.2f} BLEU")
        
        if target_lang in base_model_results:
            improvement_from_base = baseline_bleu - base_model_bleu
            improvement_pct_from_base = (improvement_from_base / base_model_bleu) * 100
            print(f"    vs Base Model:           {improvement_from_base:+.2f} BLEU ({improvement_pct_from_base:+.2f}%)")
    
    if 'experimental_stage1' in lang_results:
        stage1_bleu = lang_results['experimental_stage1']['bleu_score']
        print(f"  Experimental (Stage 1):    {stage1_bleu:.2f} BLEU")
    
    if 'experimental_stage2' in lang_results:
        experimental_bleu = lang_results['experimental_stage2']['bleu_score']
        print(f"  Experimental (Stage 2):    {experimental_bleu:.2f} BLEU")
    
    # Compare baseline vs experimental stage 2
    if 'baseline' in lang_results and 'experimental_stage2' in lang_results:
        baseline_bleu = lang_results['baseline']['bleu_score']
        experimental_bleu = lang_results['experimental_stage2']['bleu_score']
        improvement = experimental_bleu - baseline_bleu
        improvement_pct = (improvement / baseline_bleu) * 100
        
        print(f"  Improvement:               {improvement:+.2f} BLEU ({improvement_pct:+.2f}%)")
        
        if improvement > 0:
            print(f"  ✓ Sequential fine-tuning IMPROVED performance")
        elif improvement < 0:
            print(f"  ✗ Sequential fine-tuning DEGRADED performance")
        else:
            print(f"  = No significant difference")

print("\n" + "="*100)


FLORES-200 EVALUATION RESULTS (NLLB-200 + LoRA + BF16)
Target Language                   Model BLEU Score BLEU-1 BLEU-2 BLEU-3 BLEU-4     BP
        Tagalog Base Model (Pretrained)      30.74  66.25  38.88  24.90  16.34 0.9607
        Tagalog                Baseline      31.29  65.95  38.81  24.96  16.45 0.9772
        Tagalog     Experimental Stage1      30.69  66.06  38.72  24.75  16.16 0.9649
        Tagalog     Experimental Stage2      31.46  65.65  38.58  24.84  16.40 0.9871

IMPROVEMENT ANALYSIS

TAGALOG:
  Base Model (Pretrained):   30.74 BLEU
  Baseline:                  31.29 BLEU
    vs Base Model:           +0.55 BLEU (+1.78%)
  Experimental (Stage 1):    30.69 BLEU
  Experimental (Stage 2):    31.46 BLEU
  Improvement:               +0.17 BLEU (+0.54%)
  ✓ Sequential fine-tuning IMPROVED performance



## Save Results

Save evaluation results to JSON file.

In [11]:
# Save results to JSON
output_file = RESULTS_DIR / "flores200_nllb_lora_bf16_evaluation_results.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\n✓ Results saved to: {output_file}")
print(f"  Total languages evaluated: {len(all_results)}")
for target_lang, lang_results in all_results.items():
    print(f"    {target_lang.capitalize()}: {len(lang_results)} model(s)")


✓ Results saved to: ..\results\flores200_nllb_lora_bf16_evaluation_results.json
  Total languages evaluated: 1
    Tagalog: 3 model(s)


## Sample Translations

Display sample translations for qualitative analysis.

In [15]:
# Display sample translations
for target_lang, lang_results in all_results.items():
    print("\n" + "="*100)
    print(f"SAMPLE TRANSLATIONS - {target_lang.upper()}")
    print("="*100)
    
    # Show base model first if available
    if target_lang in base_model_results:
        result = base_model_results[target_lang]
        print(f"\n{'-'*100}")
        print(f"{result['model_name']}")
        print(f"BLEU Score: {result['bleu_score']:.2f}")
        print(f"{'-'*100}\n")
        
        for i, sample in enumerate(result['sample_translations'][:3], 1):
            print(f"Example {i}:")
            print(f"  Source:      {sample['source']}")
            print(f"  Reference:   {sample['reference']}")
            print(f"  Translation: {sample['translation']}")
            print()
    
    # Show fine-tuned models
    for stage, result in lang_results.items():
        print(f"\n{'-'*100}")
        print(f"{result['model_name']}")
        print(f"BLEU Score: {result['bleu_score']:.2f}")
        print(f"{'-'*100}\n")
        
        for i, sample in enumerate(result['sample_translations'][:3], 1):
            print(f"Example {i}:")
            print(f"  Source:      {sample['source']}")
            print(f"  Reference:   {sample['reference']}")
            print(f"  Translation: {sample['translation']}")
            print()
    
    print("="*100)


SAMPLE TRANSLATIONS - TAGALOG

----------------------------------------------------------------------------------------------------
Tagalog - Base Model (Pretrained)
BLEU Score: 30.74
----------------------------------------------------------------------------------------------------

Example 1:
  Source:      "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
  Reference:   "Mayroon na tayong 4 na buwang gulang na daga na hindi diabetic na dating diabetic," dagdag niya.
  Translation: "May-ari na tayo ngayon ng apat na buwan na mga mouse na hindi diabetic na dati ay diabetic", dagdag niya.

Example 2:
  Source:      Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scotia and chair of the clinical and scientific division of the Canadian Diabetes Association cautioned that the research is still in its early days.
  Reference:   Nagbabala si Dr. Ehud Ur, isang propesor sa medisina sa Dalhousie University sa Halifax, Nova S

## Summary

### What This Notebook Does

This notebook validates the trained NLLB-200 models with LoRA adapters on the FLORES-200 benchmark dataset to assess their translation quality.

### Evaluation Setup

- **Dataset:** FLORES-200 devtest split (~1000 sentence pairs per language)
- **Metrics:** BLEU score (sacrebleu implementation)
- **Models Evaluated:**
  - **Baseline:** Direct fine-tuning (e.g., en→tl only)
  - **Experimental Stage 1:** Similar language transfer (e.g., war→tl)
  - **Experimental Stage 2:** Sequential fine-tuning (war→tl THEN en→tl)

### Key Comparisons

1. **Baseline vs Experimental Stage 2:** Does similarity transfer help?
2. **Stage 1 vs Stage 2:** How much does the baseline training improve?
3. **Overall Performance:** How well do the models generalize to FLORES-200?

### Expected Findings

- If **Experimental Stage 2 > Baseline**: Sequential fine-tuning with similarity transfer is effective
- If **Baseline > Experimental Stage 2**: Direct fine-tuning is more effective
- **Stage 1 performance**: Shows how well similar language transfer works alone

### Output Files

- `results/flores200_nllb_lora_bf16_evaluation_results.json`: Complete evaluation results with sample translations

### Next Steps

1. Compare FLORES-200 results with training set validation results
2. Analyze sample translations for quality assessment
3. Identify patterns in improvements/degradations
4. Consider additional evaluation metrics (chrF, TER, etc.)